In [1]:
from pymatgen.core import Lattice, Structure
import pandas as pd
import numpy as np
import plotly as pt
import seaborn as sns
#!pip install pymatgen
#!pip install mp_api
import requests
import json
import matplotlib.pyplot as plt

In [2]:
root_folder = ""
#data_folder = "Data/" #"/content/drive/MyDrive/University/Artificial intelligence in chemistry/Perovskite project/Perovskite-liked-oxides-bandgap-prediction/Data/"
cif_folder = "Data/CIF/"
#input_dataset_path = "checkpoint_cif_embeddings_split.xlsx"
#input_dataset_path = "checkpoint_cif_embeddings_split_merged.xlsx"
#input_dataset_path = "Data/Perovskite dataset export.xlsx"
#input_dataset_path = "Mattergen dataset generation IO/dataset_1.xlsx"
input_dataset_path = "Mattergen dataset generation IO/dataset_merged.xlsx"
extend_mattergen_dataset = True
No_promoter = True

In [3]:
df = pd.read_excel(input_dataset_path)
#df = pd.read_excel(input_dataset_path, sheet_name='Photocatalytic dataset')

In [4]:
df

,Unnamed: 0,Perovskite,Hill formula,Interlayer space composition,Class,"Bandgap, eV",Materials Project ID,COD_ID,Springer_ID,Z,...,Valence Electrons Density_manual,Oxygen_count,Oxygen_concentration_manual,Oxygen_concentration_MP,Oxygen_concentration_COD,Oxygen_concentration_Springer,MP_packing_fraction,COD_packing_fraction,Springer_packing_fraction,Manual_packing_fraction
0,0,K4Nb6O17,K4 Nb6 O17,NaN,K4Nb6O17,3.50,mp-560692,1001842,-1,4.0,...,0.077010,17,0.038505,0.038482,0.040481,NaN,0.482644,0.507705,NaN,0.482925
1,1,KLaNb2O7,K1 La1 Nb2 O7,NaN,HLaNb2O7,3.20,mp-1223501,1545643,-1,8.0,...,0.086868,7,0.043434,0.040335,0.021337,NaN,0.484504,0.256300,NaN,0.521726
2,2,RbLaNb2O7,Rb1 La1 Nb2 O7,NaN,HLaNb2O7,3.35,mp-553965,-1,-1,1.0,...,0.084409,7,0.042204,0.040114,NaN,NaN,0.507344,NaN,NaN,0.533788
3,3,CsLaNb2O7,Cs1 La1 Nb2 O7,NaN,HLaNb2O7,3.30,mp-553248,2004917,-1,1.0,...,0.082082,7,0.041041,0.038958,0.041070,NaN,0.524329,0.552752,NaN,0.552364
4,4,KCa2Nb3O10,K1 Ca2 Nb3 O10,NaN,KCa2Nb3O10,3.35,mp-557195,1521061,-1,8.0,...,0.090945,10,0.045472,0.043255,0.045288,NaN,0.505552,0.529316,NaN,0.531466
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
963,1268,Na2ZrTi5O13,Na2 Zr1 Ti5 O13,NaN,NaN,3.61,mp-5449,4000748,-1,NaN,...,NaN,13,NaN,0.050630,0.050746,NaN,0.519266,0.520454,NaN,NaN
964,1269,LaFeO3,La1 Fe1 O3,NaN,NaN,2.20,mp-1078634,1526450,-1,NaN,...,NaN,3,NaN,0.051970,0.049538,NaN,0.608789,0.580299,NaN,NaN
965,1270,LaFe0.85Ti0.15O3,La1 Fe0.85 Ti0.15 O3,NaN,NaN,2.10,mp-1078634,1526450,-1,NaN,...,NaN,3,NaN,0.025985,0.012385,NaN,0.302407,0.144128,NaN,NaN
966,1271,La0.85Sr0.15FeO3,La0.85 Sr0.15 Fe1 O3,NaN,NaN,1.50,mp-1078634,1526450,-1,NaN,...,NaN,3,NaN,0.025985,0.012385,NaN,0.308150,0.146865,NaN,NaN


In [5]:
df['Perovskite'].nunique()

337

In [6]:
df = df.dropna(subset=['Log_rate'])

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 968 entries, 0 to 967
Data columns (total 84 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   Unnamed: 0                          968 non-null    int64  
 1   Perovskite                          968 non-null    object 
 2   Hill formula                        968 non-null    object 
 3   Interlayer space composition        4 non-null      object 
 4   Class                               510 non-null    object 
 5   Bandgap, eV                         885 non-null    float64
 6   Materials Project ID                968 non-null    object 
 7   COD_ID                              968 non-null    int64  
 8   Springer_ID                         968 non-null    object 
 9   Z                                   407 non-null    float64
 10  a, A                                415 non-null    float64
 11  b, A                                414 non-n

# Filters

In [8]:
print(df.shape[0])
if(No_promoter):
    print("No promoter")
    #df = df[df['Promoter'] == '-']
    df = df[(df['Promoter, w%'] == 0)]
    #df= df[df["Promoter"] == "No promoter"]
else:
    print("Pt")
    df= df[df["Promoter"] == "Pt"]
print(df.shape[0])

968
No promoter
399


In [9]:
print(df.shape[0])
df= df[df["Nitrogen"] == False]
print(df.shape[0])

399
399


In [281]:
#df = df[df["Materials Project ID"].str.contains(r"^mp-",na=False)]

In [282]:
#df_nan = df[df["MP_CIF_modifier"].isna()]

In [10]:
print(df.shape[0])
df['Alcohol_nonzero'] = df['Alcohol, %'].replace(0, np.nan)
df = df[df['Alcohol_nonzero'].notna()]
print(df.shape[0])

399
256


# Hydrogen rate standartization

In [11]:
if(No_promoter):
    df['Rate_standardized'] = df['Rate, umol/(g*h)'] / (df['Alcohol, %']*df['CatW, g/L']*df["Power, W"])
else:
    df['Rate_standardized'] = df['Rate, umol/(g*h)'] / (df['Alcohol, %']*df['CatW, g/L']*df["Power, W"]*df["Promoter, w%"])

In [12]:
df["Log_rate_standardized"] = np.log(df['Rate_standardized'])
df["Log_rate"]=df["Log_rate_standardized"]

# Remove duplicates

In [13]:
df = df.groupby('Perovskite').agg({
    'Rate_standardized': 'mean',
    'MP_CIF_modified': 'first',
    'COD_CIF_modified': 'first',
    'Springer_CIF_modified': 'first'
}).reset_index()
df

,Perovskite,Rate_standardized,MP_CIF_modified,COD_CIF_modified,Springer_CIF_modified
0,Ag2La2Ti3O10,0.006886,mp-6000,1509663,None
1,AgCa2Nb3O10,0.060778,M_MP98,M_COD45,None
2,AgLaNb2O7,0.006228,mp-1222828,1509430,None
3,AgSr2Nb3O10,0.037605,M_MP99,M_COD46,None
4,Ba5Nb4O15,0.284337,N_MP44,-1,N_Springer14
...,...,...,...,...,...
143,Zn0.83Ti0.17S,NaN,N_MP19,N_COD19,None
144,Zn0.85Ti0.15S,NaN,N_MP18,N_COD18,None
145,Zn0.9Ti0.1S,NaN,N_MP20,N_COD20,None
146,ZnGa2O4,0.008660,mp-5794,4001767,None


In [14]:
df.to_excel("checkpoint_groupby.xlsx")

# Get materials info

In [ ]:
import os
from pymatgen.io.cif import CifWriter
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
from mp_api.client import MPRester
API_KEY = ""

In [16]:
def get_material_info_from_id(MP_ID):
  output =  {"cif":"","elements":[],"pretty_formula":"","spacegroup_number":np.nan}
  file_path=cif_folder + str(MP_ID)+".cif"
  #print("Path: ",file_path)
  if os.path.exists(file_path):
    try:
      structure = Structure.from_file(file_path)
    except:
      print('ERROR: Invalid structure for ',MP_ID)
      return output
  else:
    return output

  if(structure == None):
    return output
  
  writer = CifWriter(structure)
  cif_string = str(writer)
  if not structure.is_ordered:
    print("Structure is unordered (disordered).")
    cif_string=""
  else:
    print("Structure is perfectly ordered.")


  formula_string = structure.composition.reduced_composition
  formula = str(formula_string).replace(" ","")
  print(formula)

  elements = structure.elements
  element_labels = [el.symbol for el in elements]
  print(element_labels)

  sga = SpacegroupAnalyzer(structure)
  sg_number = sga.get_space_group_number()

  for site in structure:
    if not site.is_ordered:
        print(f"Partial occupancy detected at {site.frac_coords}: {site.species}")
        return output

  #print(f"Space Group Number: {sg_number}")
  #print("CIF string: ",cif_string)
  return {"cif":cif_string,"elements":element_labels,"pretty_formula":formula,"spacegroup_number":sg_number}

In [17]:
print(get_material_info_from_id("mp-4423"))
#print(get_material_info_from_id("mp-31760"))

Structure is perfectly ordered.
La2Ti2O7
['La', 'Ti', 'O']
{'cif': "# generated using pymatgen\ndata_La2Ti2O7\n_symmetry_space_group_name_H-M   'P 1'\n_cell_length_a   7.41544296\n_cell_length_b   7.41544296\n_cell_length_c   7.41544296\n_cell_angle_alpha   60.00000000\n_cell_angle_beta   60.00000000\n_cell_angle_gamma   60.00000000\n_symmetry_Int_Tables_number   1\n_chemical_formula_structural   La2Ti2O7\n_chemical_formula_sum   'La4 Ti4 O14'\n_cell_volume   288.33429290\n_cell_formula_units_Z   2\nloop_\n _symmetry_equiv_pos_site_id\n _symmetry_equiv_pos_as_xyz\n  1  'x, y, z'\nloop_\n _atom_site_type_symbol\n _atom_site_label\n _atom_site_symmetry_multiplicity\n _atom_site_fract_x\n _atom_site_fract_y\n _atom_site_fract_z\n _atom_site_occupancy\n  La  La0  1  0.12500000  0.12500000  0.62500000  1.0\n  La  La1  1  0.12500000  0.62500000  0.12500000  1.0\n  La  La2  1  0.62500000  0.12500000  0.12500000  1.0\n  La  La3  1  0.12500000  0.12500000  0.12500000  1.0\n  Ti  Ti4  1  0.62500

In [18]:
def get_material_info_IDs(MP_ID, COD_ID, Springer_ID):
    #print("func")
    MP_info = get_material_info_from_id(MP_ID)
    if(MP_info["cif"]):
        return MP_info
    COD_info = get_material_info_from_id(COD_ID)
    if(COD_info["cif"]):
        return COD_info
    Springer_info = get_material_info_from_id(Springer_ID)
    if(Springer_info["cif"]):
        return Springer_info
    return None

In [19]:
get_material_info_IDs("mp-31760","","")

Structure is perfectly ordered.
Sr2Ta1Fe1O6
['Sr', 'Ta', 'Fe', 'O']


{'cif': "# generated using pymatgen\ndata_Sr2TaFeO6\n_symmetry_space_group_name_H-M   'P 1'\n_cell_length_a   5.66717500\n_cell_length_b   9.81577371\n_cell_length_c   9.81532669\n_cell_angle_alpha   70.70136550\n_cell_angle_beta   90.00000000\n_cell_angle_gamma   90.00000000\n_symmetry_Int_Tables_number   1\n_chemical_formula_structural   Sr2TaFeO6\n_chemical_formula_sum   'Sr8 Ta4 Fe4 O24'\n_cell_volume   515.32350959\n_cell_formula_units_Z   4\nloop_\n _symmetry_equiv_pos_site_id\n _symmetry_equiv_pos_as_xyz\n  1  'x, y, z'\nloop_\n _atom_site_type_symbol\n _atom_site_label\n _atom_site_symmetry_multiplicity\n _atom_site_fract_x\n _atom_site_fract_y\n _atom_site_fract_z\n _atom_site_occupancy\n  Sr  Sr0  1  0.48739800  0.12345700  0.62621200  1.0\n  Sr  Sr1  1  0.48739800  0.62345700  0.12621200  1.0\n  Sr  Sr2  1  0.98739800  0.87654300  0.87378800  1.0\n  Sr  Sr3  1  0.98739800  0.37654300  0.37378800  1.0\n  Sr  Sr4  1  0.01260200  0.12345700  0.12621200  1.0\n  Sr  Sr5  1  0.012

In [20]:
output_columns = [
    'material_id',	
    'formation_energy_per_atom',
    'dft_band_gap',
    'pretty_formula',
    'e_above_hull',
    'elements',
    'cif',
    'spacegroup_number',
    'azure_bulk_modulus',
    'larsen_score_2d',
    'Si_100_mismatch',
    'azure_band_gap',
    'dft_bulk_modulus',
    'dft_poisson_ratio',
    'dft_mag_density',
]
for c in output_columns:
    df[c]=None

In [21]:
df.columns

Index(['Perovskite', 'Rate_standardized', 'MP_CIF_modified',
       'COD_CIF_modified', 'Springer_CIF_modified', 'material_id',
       'formation_energy_per_atom', 'dft_band_gap', 'pretty_formula',
       'e_above_hull', 'elements', 'cif', 'spacegroup_number',
       'azure_bulk_modulus', 'larsen_score_2d', 'Si_100_mismatch',
       'azure_band_gap', 'dft_bulk_modulus', 'dft_poisson_ratio',
       'dft_mag_density'],
      dtype='object')

In [22]:
print(df.shape[0])

148


In [23]:
results=[]
for row in df.to_dict('records'):
    MP_ID = row['MP_CIF_modified']
    COD_ID = row['COD_CIF_modified']
    Springer_ID = row['Springer_CIF_modified']
    #log_rate = row['Log_rate']
    info = get_material_info_IDs(MP_ID,COD_ID,Springer_ID)
    if(info is None):
        continue
    row['cif']=info["cif"]
    row['elements']=info["elements"]
    row['pretty_formula']=info["pretty_formula"]
    row['spacegroup_number']=info["spacegroup_number"]
    results.append(row)
df = pd.DataFrame(results)

Structure is perfectly ordered.
La2Ti3Ag2O10
['La', 'Ti', 'Ag', 'O']
Structure is perfectly ordered.
Ca2Nb3Ag1O10
['Ca', 'Nb', 'Ag', 'O']
Structure is perfectly ordered.
La1Nb2Ag1O7
['La', 'Nb', 'Ag', 'O']
Structure is perfectly ordered.
Sr2Nb3Ag1O10
['Sr', 'Nb', 'Ag', 'O']
Structure is perfectly ordered.
Ba5Nb4O15
['Ba', 'Nb', 'O']
Structure is unordered (disordered).
Ba5Ta2Nb2O15
['Ba', 'Ta', 'Nb', 'O']
Partial occupancy detected at [0.66666667 0.33333333 0.89621845]: Ta0.5 Nb0.5
Structure is unordered (disordered).
Ba5Ta2Nb2O15
['Ba', 'Ta', 'Nb', 'O']
Partial occupancy detected at [0.33333333 0.66666667 0.1035    ]: Ta0.5 Nb0.5
Structure is perfectly ordered.
Ba5Ta4O15
['Ba', 'Ta', 'O']
Structure is perfectly ordered.
Ba1Nb2Bi2O9
['Ba', 'Nb', 'Bi', 'O']
Structure is perfectly ordered.
Ba1Ta2Bi2O9
['Ba', 'Ta', 'Bi', 'O']
Structure is unordered (disordered).
Ti2.5Cr0.5Bi4O12
['Ti', 'Cr', 'Bi', 'O']
Partial occupancy detected at [0.364722 0.364722 0.257035]: Ti0.83333333 Cr0.16666667
S

e:\Programming practice\Python\Perovskite\.venv\Lib\site-packages\pymatgen\core\structure.py:3112: UserWarning: Issues encountered while parsing CIF: 12 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = parser.parse_structures(primitive=primitive)[0]
e:\Programming practice\Python\Perovskite\.venv\Lib\site-packages\pymatgen\core\structure.py:3112: UserWarning: Issues encountered while parsing CIF: 16 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = parser.parse_structures(primitive=primitive)[0]
C:\Users\Nikita\AppData\Local\Temp\ipykernel_13060\3543756238.py:17: UserWarning: Site labels are not unique, which is not compliant with the CIF spec (https://www.iucr.org/__data/iucr/cifdic_html/1/cif_core.dic/Iatom_site_label.html):`['Ba0', 'Ba1', 'Ba2', 'Ba3', 'Ba4', 'Ta5', 'Ta5', 'Ta6', 'Ta6', 'Ta7', 'Ta7', 'Ta8', 'Ta8', 'O9', 'O10', 'O11', 'O12', 'O13', 'O14', 'O15', 'O16', 'O17', 'O18', 'O19', 

Structure is unordered (disordered).
Ti2.7Cr0.3Bi4O12
['Ti', 'Cr', 'Bi', 'O']
Partial occupancy detected at [0.364722 0.364722 0.257035]: Ti0.9 Cr0.1
Structure is unordered (disordered).
Ti10.8Cr1.2Bi16O48
['Ti', 'Cr', 'Bi', 'O']
Partial occupancy detected at [0.3714 0.999  0.0533]: Ti0.9 Cr0.1
Structure is unordered (disordered).
Ti2.85Cr0.15Bi4O12
['Ti', 'Cr', 'Bi', 'O']
Partial occupancy detected at [0.364722 0.364722 0.257035]: Ti0.95 Cr0.05
Structure is unordered (disordered).
Ti11.4Cr0.6Bi16O48
['Ti', 'Cr', 'Bi', 'O']
Partial occupancy detected at [0.3714 0.999  0.0533]: Ti0.95 Cr0.05
Structure is unordered (disordered).
Ti2.94Cr0.06Bi4O12
['Ti', 'Cr', 'Bi', 'O']
Partial occupancy detected at [0.364722 0.364722 0.257035]: Ti0.98 Cr0.02
Structure is unordered (disordered).
Ti11.76Cr0.24Bi16O48
['Ti', 'Cr', 'Bi', 'O']
Partial occupancy detected at [0.3714 0.999  0.0533]: Ti0.98 Cr0.02
Structure is perfectly ordered.
Ti3Bi4O12
['Ti', 'Bi', 'O']
Structure is perfectly ordered.
Ca1Nb2

C:\Users\Nikita\AppData\Local\Temp\ipykernel_13060\3543756238.py:17: UserWarning: Site labels are not unique, which is not compliant with the CIF spec (https://www.iucr.org/__data/iucr/cifdic_html/1/cif_core.dic/Iatom_site_label.html):`['Cs1', 'Nd1', 'Ta1', 'Ta1', 'O1', 'O1', 'O1', 'O1', 'O2', 'O2', 'O3']`.
  writer = CifWriter(structure)
e:\Programming practice\Python\Perovskite\.venv\Lib\site-packages\pymatgen\core\structure.py:3112: UserWarning: No structure parsed for section 1 in CIF.
'_atom_site_label'
  struct = parser.parse_structures(primitive=primitive)[0]
e:\Programming practice\Python\Perovskite\.venv\Lib\site-packages\pymatgen\io\cif.py:1057: UserWarning: No _symmetry_equiv_pos_as_xyz type key found. Spacegroup from _symmetry_space_group_name_H-M used.
  self.symmetry_operations = self.get_symops(data)  # type:ignore[assignment]
e:\Programming practice\Python\Perovskite\.venv\Lib\site-packages\pymatgen\io\cif.py:1342: UserWarning: Cannot determine chemical composition from

Structure is perfectly ordered.
Cs1Nd1Ta2O7
['Cs', 'Nd', 'Ta', 'O']
Structure is unordered (disordered).
Nd2Ti3Rb0.5H1.5O10
['Nd', 'Ti', 'Rb', 'H', 'O']
Partial occupancy detected at [0.292256 0.292256 0.      ]: Rb0.25 H0.75
Structure is perfectly ordered.
La2Ti3H2O10
['La', 'Ti', 'H', 'O']
Structure is perfectly ordered.
La2Ti3H16C4N2O10
['La', 'Ti', 'H', 'C', 'N', 'O']
Structure is perfectly ordered.
La2Ti3H14C4O12
['La', 'Ti', 'H', 'C', 'O']
Structure is perfectly ordered.
La2Ti3H24C8N2O10
['La', 'Ti', 'H', 'C', 'N', 'O']
Structure is perfectly ordered.
La2Ti3H22C8O12
['La', 'Ti', 'H', 'C', 'O']
Structure is perfectly ordered.
La2Ti3H32C12N2O10
['La', 'Ti', 'H', 'C', 'N', 'O']
Structure is perfectly ordered.
La2Ti3H30C12O12
['La', 'Ti', 'H', 'C', 'O']
Structure is perfectly ordered.
La2Ti3H40C16N2O10
['La', 'Ti', 'H', 'C', 'N', 'O']


...

C:\Users\Nikita\AppData\Local\Temp\ipykernel_13060\3543756238.py:17: UserWarning: Site labels are not unique, which is not compliant with the CIF spec (https://www.iucr.org/__data/iucr/cifdic_html/1/cif_core.dic/Iatom_site_label.html):`['Nb1', 'Nb1', 'Nb2', 'Nb2', 'Nb3', 'Nb3', 'Nb4', 'Nb4', 'Nb5', 'Nb5', 'Nb6', 'Nb6', 'Nb7', 'Nb7', 'Nb8', 'Nb8', 'Nb9', 'Nb9', 'Nb10', 'Nb10', 'Nb11', 'Nb11', 'Nb12', 'Nb12', 'Nb13', 'Nb13', 'Nb15', 'Nb14', 'Nb14', 'O1', 'O1', 'O2', 'O2', 'O3', 'O3', 'O4', 'O4', 'O5', 'O5', 'O6', 'O6', 'O7', 'O7', 'O8', 'O8', 'O9', 'O9', 'O10', 'O10', 'O11', 'O11', 'O12', 'O12', 'O13', 'O13', 'O14', 'O14', 'O15', 'O15', 'O16', 'O16', 'O17', 'O17', 'O18', 'O18', 'O19', 'O19', 'O20', 'O20', 'O21', 'O21', 'O22', 'O22', 'O23', 'O23', 'O24', 'O24', 'O25', 'O25', 'O26', 'O26', 'O27', 'O27', 'O28', 'O28', 'O29', 'O29', 'O30', 'O30', 'O31', 'O31', 'O32', 'O32', 'O33', 'O33', 'O34', 'O34', 'O35', 'O36']`.
  writer = CifWriter(structure)
C:\Users\Nikita\AppData\Local\Temp\ipykerne

Structure is perfectly ordered.
Rb1La1Nb2O7
['Rb', 'La', 'Nb', 'O']
Structure is perfectly ordered.
Rb1La1Ta2O7
['Rb', 'La', 'Ta', 'O']
Structure is perfectly ordered.
Rb1Nd1Nb2O7
['Rb', 'Nd', 'Nb', 'O']
Structure is perfectly ordered.
Rb1Nd1Ta2O7
['Rb', 'Nd', 'Ta', 'O']
Structure is perfectly ordered.
Rb1Pr1Ta2O7
['Rb', 'Pr', 'Ta', 'O']
Structure is perfectly ordered.
Rb1Sm1Ta2O7
['Rb', 'Sm', 'Ta', 'O']
Structure is perfectly ordered.
Sr1Nb2Bi2O9
['Sr', 'Nb', 'Bi', 'O']
Structure is perfectly ordered.
Sr1Ta2Bi2O9
['Sr', 'Ta', 'Bi', 'O']
Structure is perfectly ordered.
Sr1Sn1O3
['Sr', 'Sn', 'O']
Structure is perfectly ordered.
Sr1Ti1O3
['Sr', 'Ti', 'O']
Structure is perfectly ordered.
Ti1O2
['Ti', 'O']
Structure is unordered (disordered).
Ti8.5Zn41.5S50
['Ti', 'Zn', 'S']
Partial occupancy detected at [0.44687949 0.90280256 0.83674984]: Ti0.17 Zn0.83
Structure is unordered (disordered).
Ti0.34Zn1.66S2
['Ti', 'Zn', 'S']
Partial occupancy detected at [0.33333333 0.66666667 0.        ]: Ti

C:\Users\Nikita\AppData\Local\Temp\ipykernel_13060\3543756238.py:17: UserWarning: Site labels are not unique, which is not compliant with the CIF spec (https://www.iucr.org/__data/iucr/cifdic_html/1/cif_core.dic/Iatom_site_label.html):`['Zn0', 'Zn0', 'Zn1', 'Zn1', 'Zn2', 'Zn2', 'Zn3', 'Zn3', 'Zn4', 'Zn4', 'Zn5', 'Zn5', 'Zn6', 'Zn6', 'Zn7', 'Zn7', 'Zn8', 'Zn8', 'Zn9', 'Zn9', 'Zn10', 'Zn10', 'Zn11', 'Zn11', 'Zn12', 'Zn12', 'Zn13', 'Zn13', 'Zn14', 'Zn14', 'Zn15', 'Zn15', 'Zn16', 'Zn16', 'Zn17', 'Zn17', 'Zn18', 'Zn18', 'Zn19', 'Zn19', 'Zn20', 'Zn20', 'Zn21', 'Zn21', 'Zn22', 'Zn22', 'Zn23', 'Zn23', 'Zn24', 'Zn24', 'Zn25', 'Zn25', 'Zn26', 'Zn26', 'Zn27', 'Zn27', 'Zn28', 'Zn28', 'Zn29', 'Zn29', 'Zn30', 'Zn30', 'Zn31', 'Zn31', 'Zn32', 'Zn32', 'Zn33', 'Zn33', 'Zn34', 'Zn34', 'Zn35', 'Zn35', 'Zn36', 'Zn36', 'Zn37', 'Zn37', 'Zn38', 'Zn38', 'Zn39', 'Zn39', 'Zn40', 'Zn40', 'Zn41', 'Zn41', 'Zn42', 'Zn42', 'Zn43', 'Zn43', 'Zn44', 'Zn44', 'Zn45', 'Zn45', 'Zn46', 'Zn46', 'Zn47', 'Zn47', 'Zn48', 'Zn48',

Structure is unordered (disordered).
Ti7.5Zn42.5S50
['Ti', 'Zn', 'S']
Partial occupancy detected at [0.44687949 0.90280256 0.83674984]: Ti0.15 Zn0.85
Structure is unordered (disordered).
Ti0.3Zn1.7S2
['Ti', 'Zn', 'S']
Partial occupancy detected at [0.33333333 0.66666667 0.        ]: Ti0.15 Zn0.85
Structure is unordered (disordered).
Ti1Zn9S10
['Ti', 'Zn', 'S']
Partial occupancy detected at [0.44687949 0.90280256 0.83674984]: Ti0.1 Zn0.9
Structure is unordered (disordered).
Ti0.2Zn1.8S2
['Ti', 'Zn', 'S']
Partial occupancy detected at [0.33333333 0.66666667 0.        ]: Ti0.1 Zn0.9
Structure is perfectly ordered.
Zn1Ga2O4
['Zn', 'Ga', 'O']
Structure is perfectly ordered.
Zr1O2
['Zr', 'O']


C:\Users\Nikita\AppData\Local\Temp\ipykernel_13060\3543756238.py:17: UserWarning: Site labels are not unique, which is not compliant with the CIF spec (https://www.iucr.org/__data/iucr/cifdic_html/1/cif_core.dic/Iatom_site_label.html):`['Zr1', 'Zr1', 'Zr1', 'Zr1', 'Zr1', 'Zr1', 'Zr1', 'Zr1', 'O1', 'O1', 'O1', 'O1', 'O1', 'O1', 'O1', 'O1', 'O2', 'O2', 'O2', 'O2', 'O2', 'O2', 'O2', 'O2']`.
  writer = CifWriter(structure)


In [24]:
print(df.shape[0])

100


In [25]:
df

,Perovskite,Rate_standardized,MP_CIF_modified,COD_CIF_modified,Springer_CIF_modified,material_id,formation_energy_per_atom,dft_band_gap,pretty_formula,e_above_hull,elements,cif,spacegroup_number,azure_bulk_modulus,larsen_score_2d,Si_100_mismatch,azure_band_gap,dft_bulk_modulus,dft_poisson_ratio,dft_mag_density
0,Ag2La2Ti3O10,0.006886,mp-6000,1509663,None,None,None,None,La2Ti3Ag2O10,None,"[La, Ti, Ag, O]",# generated using pymatgen\ndata_La2Ti3(AgO5)2...,139,None,None,None,None,None,None,None
1,AgCa2Nb3O10,0.060778,M_MP98,M_COD45,None,None,None,None,Ca2Nb3Ag1O10,None,"[Ca, Nb, Ag, O]",# generated using pymatgen\ndata_Ca2Nb3AgO10\n...,123,None,None,None,None,None,None,None
2,AgLaNb2O7,0.006228,mp-1222828,1509430,None,None,None,None,La1Nb2Ag1O7,None,"[La, Nb, Ag, O]",# generated using pymatgen\ndata_LaNb2AgO7\n_s...,119,None,None,None,None,None,None,None
3,AgSr2Nb3O10,0.037605,M_MP99,M_COD46,None,None,None,None,Sr2Nb3Ag1O10,None,"[Sr, Nb, Ag, O]",# generated using pymatgen\ndata_Sr2Nb3AgO10\n...,123,None,None,None,None,None,None,None
4,Ba5Nb4O15,0.284337,N_MP44,-1,N_Springer14,None,None,None,Ba5Nb4O15,None,"[Ba, Nb, O]",# generated using pymatgen\ndata_Ba5Nb4O15\n_s...,164,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,SrSnO3,3.456790,mp-12867,1521093,None,None,None,None,Sr1Sn1O3,None,"[Sr, Sn, O]",# generated using pymatgen\ndata_SrSnO3\n_symm...,140,None,None,None,None,None,None,None
96,SrTiO3,0.297297,mp-4651,1512124,None,None,None,None,Sr1Ti1O3,None,"[Sr, Ti, O]",# generated using pymatgen\ndata_SrTiO3\n_symm...,140,None,None,None,None,None,None,None
97,TiO2,4.057404,mp-1245098,1010942,None,None,None,None,Ti1O2,None,"[Ti, O]",# generated using pymatgen\ndata_TiO2\n_symmet...,1,None,None,None,None,None,None,None
98,ZnGa2O4,0.008660,mp-5794,4001767,None,None,None,None,Zn1Ga2O4,None,"[Zn, Ga, O]",# generated using pymatgen\ndata_Zn(GaO2)2\n_s...,227,None,None,None,None,None,None,None


In [26]:
df.to_csv("checkpoint_material_info.csv")

In [300]:
#df = df.groupby('Perovskite')['Rate_standardized'].mean().reset_index()
#df["Log_rate_standardized"] = np.log(df['Rate_standardized'])
#df["Log_rate"]=df["Log_rate_standardized"]

In [27]:
df

,Perovskite,Rate_standardized,MP_CIF_modified,COD_CIF_modified,Springer_CIF_modified,material_id,formation_energy_per_atom,dft_band_gap,pretty_formula,e_above_hull,elements,cif,spacegroup_number,azure_bulk_modulus,larsen_score_2d,Si_100_mismatch,azure_band_gap,dft_bulk_modulus,dft_poisson_ratio,dft_mag_density
0,Ag2La2Ti3O10,0.006886,mp-6000,1509663,None,None,None,None,La2Ti3Ag2O10,None,"[La, Ti, Ag, O]",# generated using pymatgen\ndata_La2Ti3(AgO5)2...,139,None,None,None,None,None,None,None
1,AgCa2Nb3O10,0.060778,M_MP98,M_COD45,None,None,None,None,Ca2Nb3Ag1O10,None,"[Ca, Nb, Ag, O]",# generated using pymatgen\ndata_Ca2Nb3AgO10\n...,123,None,None,None,None,None,None,None
2,AgLaNb2O7,0.006228,mp-1222828,1509430,None,None,None,None,La1Nb2Ag1O7,None,"[La, Nb, Ag, O]",# generated using pymatgen\ndata_LaNb2AgO7\n_s...,119,None,None,None,None,None,None,None
3,AgSr2Nb3O10,0.037605,M_MP99,M_COD46,None,None,None,None,Sr2Nb3Ag1O10,None,"[Sr, Nb, Ag, O]",# generated using pymatgen\ndata_Sr2Nb3AgO10\n...,123,None,None,None,None,None,None,None
4,Ba5Nb4O15,0.284337,N_MP44,-1,N_Springer14,None,None,None,Ba5Nb4O15,None,"[Ba, Nb, O]",# generated using pymatgen\ndata_Ba5Nb4O15\n_s...,164,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,SrSnO3,3.456790,mp-12867,1521093,None,None,None,None,Sr1Sn1O3,None,"[Sr, Sn, O]",# generated using pymatgen\ndata_SrSnO3\n_symm...,140,None,None,None,None,None,None,None
96,SrTiO3,0.297297,mp-4651,1512124,None,None,None,None,Sr1Ti1O3,None,"[Sr, Ti, O]",# generated using pymatgen\ndata_SrTiO3\n_symm...,140,None,None,None,None,None,None,None
97,TiO2,4.057404,mp-1245098,1010942,None,None,None,None,Ti1O2,None,"[Ti, O]",# generated using pymatgen\ndata_TiO2\n_symmet...,1,None,None,None,None,None,None,None
98,ZnGa2O4,0.008660,mp-5794,4001767,None,None,None,None,Zn1Ga2O4,None,"[Zn, Ga, O]",# generated using pymatgen\ndata_Zn(GaO2)2\n_s...,227,None,None,None,None,None,None,None


In [302]:
#idx = df.groupby('Perovskite')['Alcohol_nonzero'].idxmin()  #keeps the row with the lowest value in the Alcohol_nonzero column
#idx

In [28]:
#df = df.loc[idx].copy()
#df.to_csv("checkpoint_material_info_grouped.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 20 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Perovskite                 100 non-null    object 
 1   Rate_standardized          100 non-null    float64
 2   MP_CIF_modified            95 non-null     object 
 3   COD_CIF_modified           92 non-null     object 
 4   Springer_CIF_modified      24 non-null     object 
 5   material_id                0 non-null      object 
 6   formation_energy_per_atom  0 non-null      object 
 7   dft_band_gap               0 non-null      object 
 8   pretty_formula             100 non-null    object 
 9   e_above_hull               0 non-null      object 
 10  elements                   100 non-null    object 
 11  cif                        100 non-null    object 
 12  spacegroup_number          100 non-null    int64  
 13  azure_bulk_modulus         0 non-null      object 


# Output dataset formation

In [29]:
print(df.shape[0])
df=df.dropna(subset=['cif'])
print(df.shape[0])


100
100


In [30]:
df['Log_rate']=np.log(df['Rate_standardized'])

In [31]:
output_columns.append('Log_rate')
df = df[output_columns]

In [32]:
df.shape

(100, 16)

In [33]:
df = df.sample(frac=1).reset_index(drop=True)

In [34]:
df.to_csv("mattergen_dataset_for_fine_tuning.csv", index=False)

In [ ]:
ss/0

NameError: name 'ss' is not defined

In [ ]:
name = "train"

In [ ]:
df_ = pd.read_csv(f"Data/Mattergen_dataset/processed/{name}.csv")
df_

,Unnamed: 0,material_id,formation_energy_per_atom,dft_band_gap,pretty_formula,e_above_hull,elements,cif,spacegroup_number,azure_bulk_modulus,larsen_score_2d,Si_100_mismatch,azure_band_gap,dft_bulk_modulus,dft_poisson_ratio,dft_mag_density,Log_rate
0,0,0,NaN,NaN,La1Ti1H12C4N1O4,NaN,"['La', 'Ti', 'H', 'C', 'N', 'O']",# generated using pymatgen\ndata_LaTiH12C4NO4\...,11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.596764
1,1,1,NaN,NaN,Nd2Ti3H16C4N2O10,NaN,"['Nd', 'Ti', 'H', 'C', 'N', 'O']",# generated using pymatgen\ndata_Nd2Ti3H16C4(N...,12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.854861
2,2,2,NaN,NaN,La1Ti1H1O4,NaN,"['La', 'Ti', 'H', 'O']",# generated using pymatgen\ndata_LaTiHO4\n_sym...,129,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.924004
3,3,3,NaN,NaN,La1Nb2H1O7,NaN,"['La', 'Nb', 'H', 'O']",# generated using pymatgen\ndata_LaNb2HO7\n_sy...,123,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.613204
4,4,4,NaN,NaN,Nd1Ti1H7C2O5,NaN,"['Nd', 'Ti', 'H', 'C', 'O']",# generated using pymatgen\ndata_NdTiH7C2O5\n_...,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.326302
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
60,61,61,NaN,NaN,Ca2Nb3H1O10,NaN,"['Ca', 'Nb', 'H', 'O']",# generated using pymatgen\ndata_Ca2Nb3HO10\n_...,123,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.028544
61,62,62,NaN,NaN,Rb1Sr2Nb3O10,NaN,"['Rb', 'Sr', 'Nb', 'O']",# generated using pymatgen\ndata_RbSr2Nb3O10\n...,123,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-2.851554
62,63,63,NaN,NaN,Nd1Ti1H8C2N1O4,NaN,"['Nd', 'Ti', 'H', 'C', 'N', 'O']",# generated using pymatgen\ndata_NdTiH8C2NO4\n...,11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.951608
63,64,64,NaN,NaN,Rb1La1Nb2O7,NaN,"['Rb', 'La', 'Nb', 'O']",# generated using pymatgen\ndata_RbLaNb2O7\n_s...,74,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-3.798880


In [ ]:
df_ = df_.head(95)
df_

,Unnamed: 0,material_id,formation_energy_per_atom,dft_band_gap,pretty_formula,e_above_hull,elements,cif,spacegroup_number,azure_bulk_modulus,larsen_score_2d,Si_100_mismatch,azure_band_gap,dft_bulk_modulus,dft_poisson_ratio,dft_mag_density,Log_rate
0,0.0,0.0,NaN,NaN,Rb1Sr2Nb3O10,NaN,"['Rb', 'Sr', 'Nb', 'O']",# generated using pymatgen\ndata_RbSr2Nb3O10\n...,123.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-2.851554
1,1.0,1.0,NaN,NaN,Nd1Ti1H9C3O5,NaN,"['Nd', 'Ti', 'H', 'C', 'O']",# generated using pymatgen\ndata_NdTiH9C3O5\n_...,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.109061
2,2.0,2.0,NaN,NaN,Ca2Nb3H1O10,NaN,"['Ca', 'Nb', 'H', 'O']",# generated using pymatgen\ndata_Ca2Nb3HO10\n_...,123.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.028544
3,3.0,3.0,NaN,NaN,Ca2Nb3H12C2N2O10,NaN,"['Ca', 'Nb', 'H', 'C', 'N', 'O']",# generated using pymatgen\ndata_Ca2Nb3H12C2(N...,25.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.654088
4,4.0,4.0,NaN,NaN,La2Ti2O7,NaN,"['La', 'Ti', 'O']",# generated using pymatgen\ndata_La2Ti2O7\n_sy...,227.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-7.927406
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
90,90.0,90.0,NaN,NaN,Nd1Ti1H16C6N1O4,NaN,"['Nd', 'Ti', 'H', 'C', 'N', 'O']",# generated using pymatgen\ndata_NdTiH16C6NO4\...,11.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.644755
91,91.0,91.0,NaN,NaN,La2Ti3H22C8O12,NaN,"['La', 'Ti', 'H', 'C', 'O']",# generated using pymatgen\ndata_La2Ti3H22(C2O...,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.473433
92,92.0,92.0,NaN,NaN,K1La1Nb2O7,NaN,"['K', 'La', 'Nb', 'O']",# generated using pymatgen\ndata_KLaNb2O7\n_sy...,38.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-3.525257
93,93.0,93.0,NaN,NaN,K2La2Ti3O10,NaN,"['K', 'La', 'Ti', 'O']",# generated using pymatgen\ndata_K2La2Ti3O10\n...,139.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.095464


In [ ]:
df.to_csv(f"Data/Mattergen_dataset/processed/{name}.csv")